# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lokeshtiwari723/Proto-ex/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1: Content Lifecycle — Growing vs Declining

The paper reports that growing pages were younger on average than declining pages: about 185 days versus 228 days. It also reports similar average word counts between the two groups. The methodology question I would ask is: how much of this observed difference is associated with content age itself, and how much could reflect other factors such as visibility, page type, or publishing history? The comparison is descriptive, so it does not by itself establish that age causes decline.

### Finding 2: Refreshing Pages Actually Works

The paper reports positive median impression lift for refreshed older pages and states that 7 of 9 strata showed statistically significant refresh lift. The methodology question I would ask is: how was the refresh comparison constructed, and what other differences between refreshed and stale pages could explain part of the observed lift? A comparison can show an association, but the design needs to be examined before treating the result as causal.

### Why these questions matter for my model

My Week-5 Logistic Regression model is also a decision-support analysis. I need to check whether the validation design supports the claim I make from the model. I will therefore re-run the model under an honest grouped split, check the feature set for leakage, inspect concrete errors, and rewrite the claim using only observed and measured evidence.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1: Verification checklist

paper_findings = {
    "Finding 1": "Content Lifecycle — Growing vs Declining",
    "Finding 2": "Refreshing Pages Actually Works"
}

methodology_questions = {
    "Finding 1": "Could other factors explain the observed age difference?",
    "Finding 2": "How was the refresh comparison constructed, and could other differences explain the observed lift?"
}

print("Paper findings covered:")
for name, finding in paper_findings.items():
    print(f"- {name}: {finding}")

print("\nMethodology questions covered:")
for name, question in methodology_questions.items():
    print(f"- {name}: {question}")

print("\nSection 1 checklist:")
print("Two findings:", len(paper_findings) == 2)
print("Two methodology questions:", len(methodology_questions) == 2)
print("Causal language treated cautiously: True")
print("Section 1: READY")

Paper findings covered:
- Finding 1: Content Lifecycle — Growing vs Declining
- Finding 2: Refreshing Pages Actually Works

Methodology questions covered:
- Finding 1: Could other factors explain the observed age difference?
- Finding 2: How was the refresh comparison constructed, and could other differences explain the observed lift?

Section 1 checklist:
Two findings: True
Two methodology questions: True
Causal language treated cautiously: True
Section 1: READY


## 2. My model under an honest split (before/after)

I compared the same Logistic Regression model under two split designs.

Before: a random row-level split, where the same client can appear in both training and test data.

After: a grouped split by client, so no client appears in both training and test data.

I kept the feature set, March label construction, model type, and Precision@50 metric the same. The grouped split is the stricter validation design because it tests whether the model can rank pages for clients it did not see during training.

The purpose is decision support: measure how the validation design changes the observed result, not to claim causal performance.

In [11]:
# ML-09 setup: reconnect to the warehouse

import os
import duckdb
import pandas as pd
import numpy as np

# Load the Hugging Face token from Colab Secrets
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

print("HF_TOKEN loaded:", HF_TOKEN is not None)

# Create DuckDB connection
con = duckdb.connect()

print("DuckDB connection ready:", con is not None)

HF_TOKEN loaded: True
DuckDB connection ready: True


In [14]:
from huggingface_hub import HfApi
import duckdb

api = HfApi(token=HF_TOKEN)

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"
DIM = f"read_parquet('{REL}/dim_content.parquet')"

print("Warehouse connection ready")
print("Feature window: February 2026")
print("Label window: March 2026")

Warehouse connection ready
Feature window: February 2026
Label window: March 2026


In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 2: Before/after validation comparison
# BEFORE = random row-level split
# AFTER  = grouped split by client

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


# ------------------------------------------------------------
# 1. Rebuild the same modeling dataset used in Week 5
# ------------------------------------------------------------

features = con.sql(f"""
WITH feb_page AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS gsc_avg_position
    FROM {FEB}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.gsc_impressions,
    f.gsc_clicks,
    f.gsc_avg_position,
    d.word_count,
    d.content_created_date
FROM feb_page AS f
LEFT JOIN {DIM} AS d
  ON f.client_hash_id = d.client_hash_id
 AND f.content_hash_id = d.content_hash_id
""").df()


label_df = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_clicks) AS march_clicks,
    SUM(gsc_impressions) AS march_impressions
FROM {MAR}
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
HAVING SUM(gsc_impressions) >= 100
""").df()

label_df["march_ctr"] = (
    label_df["march_clicks"] / label_df["march_impressions"]
)

ctr_cutoff = label_df["march_ctr"].median()

label_df["label"] = (
    label_df["march_ctr"] > ctr_cutoff
).astype(int)


model_df = features.merge(
    label_df[
        [
            "client_hash_id",
            "content_hash_id",
            "march_ctr",
            "label"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)


model_df["content_created_date"] = pd.to_datetime(
    model_df["content_created_date"],
    errors="coerce"
)

decision_date = pd.Timestamp("2026-02-28")

model_df["content_age_days"] = (
    decision_date - model_df["content_created_date"]
).dt.days


feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "word_count",
    "content_age_days"
]

model_df = model_df.dropna(
    subset=feature_cols + ["label"]
).copy()


print("Rows available for modeling:", len(model_df))
print("Unique client-content pairs:",
      model_df[["client_hash_id", "content_hash_id"]].drop_duplicates().shape[0])
print("Duplicate page records:",
      model_df.duplicated(
          subset=["client_hash_id", "content_hash_id"]
      ).sum())
print("March CTR median cutoff:", round(ctr_cutoff, 6))


# ------------------------------------------------------------
# 2. Helper function
# ------------------------------------------------------------

def precision_at_k(y_true, scores, k=50):
    ranked = np.argsort(-scores)[:k]
    return float(np.mean(np.asarray(y_true)[ranked]))


def run_logistic(train_df, test_df):

    X_train = train_df[feature_cols]
    y_train = train_df["label"]

    X_test = test_df[feature_cols]
    y_test = test_df["label"]

    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ])

    pipe.fit(X_train, y_train)

    scores = pipe.predict_proba(X_test)[:, 1]

    p50 = precision_at_k(
        y_test.values,
        scores,
        k=50
    )

    return pipe, scores, p50


# ------------------------------------------------------------
# 3. BEFORE: random row-level split
# ------------------------------------------------------------

train_before, test_before = train_test_split(
    model_df,
    test_size=0.20,
    random_state=42,
    stratify=model_df["label"]
)

model_before, scores_before, p50_before = run_logistic(
    train_before,
    test_before
)

before_client_overlap = len(
    set(train_before["client_hash_id"])
    & set(test_before["client_hash_id"])
)


# ------------------------------------------------------------
# 4. AFTER: grouped split by client
# ------------------------------------------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        model_df,
        model_df["label"],
        groups=model_df["client_hash_id"]
    )
)

train_after = model_df.iloc[train_idx].copy()
test_after = model_df.iloc[test_idx].copy()

model_after, scores_after, p50_after = run_logistic(
    train_after,
    test_after
)

after_client_overlap = len(
    set(train_after["client_hash_id"])
    & set(test_after["client_hash_id"])
)


# ------------------------------------------------------------
# 5. Week-4 baseline on the same test sets
# ------------------------------------------------------------

def add_baseline_score(df):
    out = df.copy()

    visible = out["gsc_impressions"] >= 100
    stale = out["content_age_days"] >= 180

    out["action_score"] = (
        visible.astype(int)
        + stale.astype(int)
    )

    return out


test_before_base = add_baseline_score(test_before)
test_after_base = add_baseline_score(test_after)


baseline_before_ranked = test_before_base.sort_values(
    ["action_score", "gsc_impressions"],
    ascending=[False, False]
)

baseline_after_ranked = test_after_base.sort_values(
    ["action_score", "gsc_impressions"],
    ascending=[False, False]
)


baseline_p50_before = precision_at_k(
    baseline_before_ranked["label"].values,
    np.arange(len(baseline_before_ranked), 0, -1),
    k=50
)

baseline_p50_after = precision_at_k(
    baseline_after_ranked["label"].values,
    np.arange(len(baseline_after_ranked), 0, -1),
    k=50
)


# ------------------------------------------------------------
# 6. Print comparison
# ------------------------------------------------------------

comparison = pd.DataFrame({
    "Validation": [
        "Before: random row split",
        "After: grouped by client"
    ],
    "Train rows": [
        len(train_before),
        len(train_after)
    ],
    "Test rows": [
        len(test_before),
        len(test_after)
    ],
    "Client overlap": [
        before_client_overlap,
        after_client_overlap
    ],
    "Baseline Precision@50": [
        baseline_p50_before,
        baseline_p50_after
    ],
    "Logistic Regression Precision@50": [
        p50_before,
        p50_after
    ]
})

display(comparison)

print("\nBEFORE random split client overlap:", before_client_overlap)
print("AFTER grouped split client overlap:", after_client_overlap)

print(
    "\nMeasured Precision@50 change:",
    round(p50_before, 4),
    "->",
    round(p50_after, 4)
)

print(
    "\nGrouped split client overlap check:",
    "PASSED" if after_client_overlap == 0 else "FAILED"
)

Rows available for modeling: 58279
Unique client-content pairs: 58279
Duplicate page records: 0
March CTR median cutoff: 0.001238


,Validation,Train rows,Test rows,Client overlap,Baseline Precision@50,Logistic Regression Precision@50
0,Before: random row split,46623,11656,33,0.84,0.98
1,After: grouped by client,47834,10445,0,0.76,1.00



BEFORE random split client overlap: 33
AFTER grouped split client overlap: 0

Measured Precision@50 change: 0.98 -> 1.0

Grouped split client overlap check: PASSED


## 3. Leakage audit

I checked the final model feature set against fields that are only known after the decision point, fields derived from the March label window, and identifier fields.

The final features are:

- gsc_impressions
- gsc_clicks
- gsc_avg_position
- word_count
- content_age_days

The March outcome fields and label-derived fields are not used as model features.

The audit is intended to verify that the measured ranking result is based on information available at the February decision point.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3: Leakage audit

final_features = set(feature_cols)

prohibited_fields = {
    # Future / label-window fields
    "march_clicks",
    "march_impressions",
    "march_ctr",
    "label",

    # Known leakage-prone starter fields
    "trend_direction",
    "trend_pct",

    # Identifiers
    "client_hash_id",
    "content_hash_id"
}

leakage_hits = final_features & prohibited_fields

print("Final model features:")
for col in feature_cols:
    print("-", col)

print("\nProhibited fields checked:")
for col in sorted(prohibited_fields):
    print("-", col)

print("\nUnexpected feature overlap:", leakage_hits)

if len(leakage_hits) == 0:
    print("\nLEAKAGE AUDIT: PASSED")
else:
    print("\nLEAKAGE AUDIT: FAILED")

# Additional checks
print("\nClient overlap in grouped split:", after_client_overlap)

if after_client_overlap == 0:
    print("Grouped split check: PASSED")
else:
    print("Grouped split check: FAILED")

# Confirm the label is not in the feature matrix
feature_matrix_columns = set(feature_cols)

print(
    "\nLabel included as a feature:",
    "label" in feature_matrix_columns
)

print(
    "March CTR included as a feature:",
    "march_ctr" in feature_matrix_columns
)

## 4. Claim rewrite

### Original claim

The Logistic Regression model predicts which pages will have better future search performance and performs better than the baseline.

### Safer claim

On the held-out test data, Logistic Regression measured a Precision@50 that can be compared with the Week-4 action-score baseline. Under the client-grouped split, the result is evidence of a measured ranking signal in this test setup and can support decision-making about which pages to review first.

This is directional and decision-support evidence only. It does not establish causation, guarantee future performance, or show that the model will perform the same way on other clients or time periods.

The grouped validation, leakage audit, and error review are therefore part of the evidence needed before making a broader claim.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4: Print the measured results used for the claim

print("Claim evidence from this notebook")
print("----------------------------------")

print(
    "Random row-level split Logistic Regression Precision@50:",
    round(p50_before, 4)
)

print(
    "Client-grouped split Logistic Regression Precision@50:",
    round(p50_after, 4)
)

print(
    "Client-grouped split baseline Precision@50:",
    round(baseline_p50_after, 4)
)

print(
    "Client overlap in grouped split:",
    after_client_overlap
)

print(
    "Leakage overlap:",
    leakage_hits
)

print("\nSafe claim:")
print(
    f"On the held-out client-grouped test split, "
    f"Logistic Regression measured Precision@50 of {p50_after:.2f}, "
    f"compared with {baseline_p50_after:.2f} for the Week-4 action-score baseline. "
    f"This is measured evidence of ranking signal in this test setup "
    f"and should be treated as decision-support evidence, not a causal or guaranteed result."
)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.